# Exercise 3.2 Work with SAP HANA Cloud Knowledge Graph Engine from LangChain

In this exercise, you will use **LangChain** — a Python framework for building AI-powered applications — to connect SAP HANA Cloud to a large language model (LLM).

The database contains an **RDF Knowledge Graph** called `OBJECT_INDEX_RDF_GRAPH`, which you created in Exercise 1. An RDF graph stores data as a network of connected facts (e.g. *"table RESULTS has a column raceId"*). Querying it requires **SPARQL**, a graph query language similar in spirit to SQL.

Instead of writing SPARQL by hand, you will ask questions in plain English and let the LLM translate them into SPARQL queries automatically.

## Connect to SAP HANA Cloud

First, establish a connection to the SAP HANA Cloud database. The connection credentials (password, host address, port) are loaded from a `.env` file using `dotenv` — this avoids hardcoding sensitive values in the notebook.

Replace `000` in the user name with your personal student number (e.g. `DA263_007`). The `connection` object returned by `dbapi.connect()` will be reused in all subsequent steps.

In [1]:
import os
from dotenv import load_dotenv
from hdbcli import dbapi

load_dotenv()

connection = dbapi.connect(
    user="DA263_000", # (replace 000 with your student number, e.g. 001)
    password=os.environ.get("HANA_DB_PASSWORD"),
    address=os.environ.get("HANA_DB_ADDRESS"),
    port=os.environ.get("HANA_DB_PORT"),
    # sslValidateCertificate = False
)

print('HANA Cloud connected:', connection.isconnected())

Error: (10, "authentication failed: Detailed info for this error can be found with correlation ID '847AA05A4AA2B548BE51646A0F900ECE'")

## Create a Graph

Now wrap the HANA graph in a LangChain `HanaRdfGraph` object. This gives LangChain a handle to query the graph.

Setting `auto_extract_ontology=True` tells LangChain to automatically read the graph's **ontology** — the set of classes, properties, and relationships that describe what kind of data is in the graph (think of it as the schema). The LLM will use this ontology later to generate correct SPARQL queries, so it knows what terms and structures exist in the graph.

Again, replace `000` in the graph URI with your student number.

In [2]:
from langchain_hana import HanaRdfGraph

oi_graph = HanaRdfGraph(
    connection=connection,
    graph_uri="DA263_000.OBJECT_INDEX_RDF_GRAPH", # (replace 000 with your student number, e.g. 001)
    auto_extract_ontology=True,
)

## Run a SPARQL Query

Before using the LLM, run a manual SPARQL query to verify the graph is accessible and see what's in it.

The query uses three **PREFIX** declarations — these are shorthand aliases for long URI namespaces, similar to `import` statements in Python. The `WHERE` clause then selects all nodes of type `te:DatabaseObject` and retrieves their labels. The result shows the tables and views that exist in the graph.

In [4]:
query = """
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
	PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
	PREFIX te: <https://example.org/hana-metadata#>
	SELECT *
	FROM <DA263_000.OBJECT_INDEX_RDF_GRAPH>
	WHERE {
	  ?s rdf:type te:DatabaseObject.
		?s rdfs:label ?l .
	}
    """
results = oi_graph.query(
    query=query
)
print(results)

s,l
https://example.org/hana-metadata#objectOID/166699,CONSTRUCTORS
https://example.org/hana-metadata#objectOID/166777,RACES
https://example.org/hana-metadata#objectOID/166719,DRIVERS
https://example.org/hana-metadata#objectOID/166848,STATUS
https://example.org/hana-metadata#objectOID/166821,SEASONS
https://example.org/hana-metadata#objectOID/166827,SPRINTRESULTS
https://example.org/hana-metadata#objectOID/166740,LAPTIMES
https://example.org/hana-metadata#objectOID/166673,CIRCUITS
https://example.org/hana-metadata#objectOID/166753,PITSTOPS
https://example.org/hana-metadata#objectOID/166732,DRIVERSTANDINGS
https://example.org/hana-metadata#objectOID/166708,CONSTRUCTORSTANDINGS
https://example.org/hana-metadata#objectOID/166687,CONSTRUCTORRESULTS
https://example.org/hana-metadata#objectOID/167150,V_RESULTS
https://example.org/hana-metadata#objectOID/166796,RESULTS
https://example.org/hana-metadata#objectOID/166759,QUALIFYING



## Provide an LLM

To translate natural language questions into SPARQL, we need an LLM. Here we initialize **GPT-4.1** through the **SAP Generative AI Hub** — the same setup used in Exercise 3.1.

The `proxy_client` routes requests through SAP's AI infrastructure, so you don't need an OpenAI account. The resulting `llm` object is a standard LangChain chat model and will be passed into the query chain in the next step.

In [5]:
from gen_ai_hub.proxy.langchain import ChatOpenAI
from gen_ai_hub.proxy import get_proxy_client

proxy_client = get_proxy_client('gen-ai-hub')

llm = ChatOpenAI(proxy_model_name='gpt-4.1', proxy_client=proxy_client)

print("LLM model:", llm.model_name)

LLM model: gpt-4.1


## Create a SPARQL QA Chain for Natural-Language-to-SPARQL

Now wire the LLM and the graph together using `HanaSparqlQAChain`. In LangChain, a **chain** is a sequence of steps that run in order — here the steps are: receive a natural language question → ask the LLM to write a SPARQL query using the ontology → execute the query on HANA → ask the LLM to summarize the results in plain English.

`allow_dangerous_requests=True` is required because the chain executes queries against a live database. Setting `verbose=True` will print each intermediate step so you can see exactly what SPARQL the LLM produces.

In [6]:
from langchain_hana import HanaSparqlQAChain

qa_chain = HanaSparqlQAChain.from_llm(
    llm=llm, 
    graph=oi_graph, 
    allow_dangerous_requests=True, 
    verbose=True
)

Run a natural language query against the RDF graph. The chain handles everything automatically: it translates your question into SPARQL, executes it on HANA, and returns a human-readable answer.

Several example queries are commented out — feel free to try them. The verbose output below will show you the generated SPARQL and the raw query results before the final answer is produced.

In [7]:
# query = "which columns does V_RESULTS have"
# query = "which tables have a columns with datatype INTEGER"
# query = "which tables have a columns raceId"
# query = "which type of database objects are in the system"

query = "on which tables does the view V_RESULTS depend upon"
# Expected result: 
# The view V_RESULTS depends upon the following tables: RESULTS, CONSTRUCTORS, RACES, DRIVERS, and STATUS.

output = qa_chain.invoke(
    input=query
    )

print(output["result"])



> Entering new HanaSparqlQAChain chain...
Generated SPARQL:
```sparql
PREFIX hana: <https://example.org/hana-metadata#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?table ?tableLabel
WHERE {
    ?view rdf:type hana:DatabaseObject .
    ?view hana:schemaName ?schemaName .
    ?view rdfs:label ?viewLabel .
    FILTER(?viewLabel = "V_RESULTS")
    ?view hana:hasDependency ?table .
    ?table rdf:type hana:DatabaseObject .
    ?table hana:objectType ?objectType .
    FILTER(?objectType = "TABLE")
    ?table rdfs:label ?tableLabel .
}
```
Final SPARQL:

PREFIX hana: <https://example.org/hana-metadata#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?table ?tableLabel

FROM <DA263_000.OBJECT_INDEX_RDF_GRAPH>
WHERE {
    ?view rdf:type hana:DatabaseObject .
    ?view hana:schemaName ?schemaName .
    ?view rdfs:label ?viewLabel .
    FILTER(?viewLa

## Create a SPARQL QA Agent

So far the chain followed a fixed sequence of steps. Now try the **agent-based approach**, which is more flexible.

In LangChain, an **agent** is an LLM that can decide for itself which tools to call and in what order. The `HanaSparqlQAAgent` has access to two tools: one to fetch the ontology and one to execute SPARQL queries. For each question it reasons step-by-step, choosing when to retrieve the ontology and when to run a query — and it can retry with a corrected query if something goes wrong.

Compared to the chain, the agent can handle more complex, multi-step questions. You will see this reasoning process printed in the output as a sequence of tool calls and responses.

In [8]:
from langchain_hana import HanaSparqlQAAgent

agent = HanaSparqlQAAgent.create_agent(
    graph=oi_graph, 
    model=llm
    )

In [9]:
# question = "whcich table contains more columns than the table RACES"
# question = "which of the tables used in V_RESULTS have an association to each other"
question = "which of the tables used in V_RESULTS has an association to CONSTRUCTORS"

for event in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

which of the tables used in V_RESULTS has an association to CONSTRUCTORS
================================== Ai Message ==================================
Tool Calls:
  retrieve_ontology (call_BPAdOxobRwoIR5TKpkYcy6Z0)
 Call ID: call_BPAdOxobRwoIR5TKpkYcy6Z0
  Args:
================================== Ai Message ==================================
Tool Calls:
  retrieve_ontology (call_BPAdOxobRwoIR5TKpkYcy6Z0)
 Call ID: call_BPAdOxobRwoIR5TKpkYcy6Z0
  Args:
================================= Tool Message =================================
Name: retrieve_ontology

Ontology Information:
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

rdfs:label a owl:DatatypeProperty ;
    rdfs:label "label" ;
    rdfs:domain <https://example.org/hana-metadata#Column>,
        <https://example.org/hana-metadata#DatabaseObj